# 🍊 YOLOv8/11 Image Classification - Optimized Drive-First Edition

**Cấu hình tối ưu (Đã được khôi phục & Điều chỉnh cho Phân loại):**
1. **An toàn dữ liệu**: Lưu checkpoint liên tục lên Drive, tự động RESUME (tiếp tục) nếu Colab bị ngắt.
2. **Tối ưu VRAM**: Tính toán tự động Batch Size thông minh qua `get_optimal_batch()`.
3. **SSD Cache & Workers**: Tự copy data vào ổ SSD nội bộ bộ nhớ Colab và dùng RAM cache.
4. **Fail-safe**: Tạo Dummy Data khẩn cấp để đảm bảo luôn chạy qua mọi Cell trong Runtime (tránh lỗi ngắt mạch).

In [ ]:
# ============================================================
# Cell 0: Mount Google Drive - TRẠM LƯU TRỮ CHÍNH
# ============================================================
from google.colab import drive
import os

drive.mount('/content/drive')

# Thư mục gốc trên Drive chứa các folder con: cam, chanh, quyt
DRIVE_SOURCE = '/content/drive/MyDrive/PBL5'
DRIVE_ROOT = '/content/drive/MyDrive/YOLO_PBL5_Classification'
os.makedirs(DRIVE_ROOT, exist_ok=True)

print(f"✅ Nguồn ảnh phân loại (Source): {DRIVE_SOURCE}")
print(f"✅ Trạm lưu trữ mô hình (Root): {DRIVE_ROOT}")

In [ ]:
# ============================================================
# Cell 1: Cài đặt thư viện & Định chuẩn VRAM
# ============================================================
!pip install ultralytics split-folders -q

import torch
from ultralytics import YOLO

def get_optimal_batch():
    if not torch.cuda.is_available():
        return 4
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    # Mô hình Classification tốn ít VRAM hơn nên có thể nâng batch size cao hơn
    if vram >= 14: return 64
    if vram >= 10: return 32
    return 16

BATCH_SIZE = get_optimal_batch()
DEVICE = 0 if torch.cuda.is_available() else 'cpu'

print(f"✅ Thiết bị: {torch.cuda.get_device_name(0) if DEVICE==0 else 'CPU'}")
print(f"📦 Batch Size tối ưu tự động cho VRAM: {BATCH_SIZE}")

In [ ]:
# ============================================================
# Cell 2: Chuẩn bị Dataset Phân Loại (Có Dummy Khôi Phục)
# ============================================================
import os
import shutil
import splitfolders
import numpy as np
from PIL import Image

LOCAL_DATASET = '/content/dataset'
CLASS_NAMES = ['cam', 'chanh', 'quyt']

if os.path.exists(LOCAL_DATASET): shutil.rmtree(LOCAL_DATASET)

if os.path.exists(DRIVE_SOURCE) and any(os.path.isdir(os.path.join(DRIVE_SOURCE, c)) for c in CLASS_NAMES):
    print(f"📂 Tìm thấy Dataset thật trên Drive: {DRIVE_SOURCE}")
    print("⏳ Đang copy và chia tập train/val trực tiếp sang SSD của Colab (Để tránh kẹt IO Drive trên mạng)...")
    
    try:
        splitfolders.ratio(DRIVE_SOURCE, output=LOCAL_DATASET, seed=42, ratio=(0.8, 0.2))
        print(f"✅ Đã chuẩn bị Dataset Phân loại ảnh thành công tại: {LOCAL_DATASET}")
    except Exception as e:
        print(f"⚠️ Lỗi khi split-folders (có thể do các ảnh hỏng hặc thiếu dữ liệu tại {DRIVE_SOURCE}): {e}")
else:
    print(f"⚠️ Không thấy folder ảnh hiện hữu trên {DRIVE_SOURCE}.")
    print("⏳ [BẢN SỬA LỖI FAILSAFE] Đang tạo Dummy Dataset để tránh bể luồng mã nếu Run All...")
    
    for split_dir in ['train', 'val']:
        for cls in CLASS_NAMES:
            os.makedirs(f'{LOCAL_DATASET}/{split_dir}/{cls}', exist_ok=True)
            for i in range(5):
                img_arr = np.random.randint(0, 255, (224, 224, 3), dtype=np.uint8)
                Image.fromarray(img_arr).save(f'{LOCAL_DATASET}/{split_dir}/{cls}/dummy_{i}.jpg')
    print("✅ Đã sinh dữ liệu Rác/Dummy thành công.")

In [ ]:
# ============================================================
# Cell 3: Training Phân loại với TOÀN BỘ TỐI ƯU HÓA TỪ BẢN GỐC
# ============================================================
import os

PROJECT_PATH = os.path.join(DRIVE_ROOT, 'train_results')
RUN_NAME     = 'fruit_cls_v1'

LAST_CKPT = os.path.join(PROJECT_PATH, RUN_NAME, 'weights', 'last.pt')
RESUME    = os.path.exists(LAST_CKPT)

# Resume nếu có file checkpoint dang dở, nếu không sẽ train mới yolo model dành cho classification (-cls)
model = YOLO(LAST_CKPT if RESUME else 'yolov8n-cls.pt')

print(f"🚀 Trạng thái: {'TIẾP TỤC Train Từ Checkpoint Cũ (Resume)' if RESUME else 'Khởi tạo huấn luyện Từ Đầu'}")

results = model.train(
    data        = LOCAL_DATASET,
    epochs      = 50,          # PBL5 yêu cầu đủ epoch để model không bị ngu nên giữ mức 50-100
    imgsz       = 224,         # Image Classification chỉ tốn imgsz 224 là đủ accuracy xịn xò
    batch       = BATCH_SIZE,  # Tự động từ hàm RAM Cell 1
    project     = PROJECT_PATH,
    name        = RUN_NAME,
    resume      = RESUME,      # An tâm khi colab disconnect giữa ngày
    exist_ok    = True,
    cache       = True,        # TỐI ƯU RAM: Nạp luôn Data về Ô chứa RAM chính Colab (Disk IO -> Memory)
    save_period = 5,           # TỐI ƯU DRIVE: Backup trực tiếp mỗi 5 Epoch lên Drive
    patience    = 20,          # TỐI ƯU THỜI GIAN: Dừng sớm nếu loss ko giảm nghĩa là sẽ k ép thiếu
    workers     = 2,           # Sử dụng Multi-Thread I/O Python
    device      = DEVICE
)

In [ ]:
# ============================================================
# Cell 4: Đánh giá hệ thống Metrics & Draw (Đảm bảo Robust)
# ============================================================
from IPython.display import Image, display
import os

BEST_MODEL_PATH = os.path.join(PROJECT_PATH, RUN_NAME, 'weights', 'best.pt')
RESULTS_PNG     = os.path.join(PROJECT_PATH, RUN_NAME, 'results.png')

if os.path.exists(BEST_MODEL_PATH):
    print("📊 Đang chạy Validations (Kháng ngừ dự kiến mô hình)...")
    val_model = YOLO(BEST_MODEL_PATH)
    metrics = val_model.val(data=LOCAL_DATASET, device=DEVICE)
    
    if os.path.exists(RESULTS_PNG):
        print("📈 Biểu đồ Tỷ lệ Phân loại Accuracy:")
        display(Image(filename=RESULTS_PNG, width=800))
else:
    val_model = None
    print("❌ Lỗi hiệu năng: Không có tham chiếu BEST_MODEL")

In [ ]:
# ============================================================
# Cell 5: Testing trực quan bằng ngữ cảnh Global Variables
# ============================================================
import glob
import os
from IPython.display import Image, display
from ultralytics import YOLO

BEST_MODEL_PATH = os.path.join(PROJECT_PATH, RUN_NAME, 'weights', 'best.pt')
if ('val_model' not in globals() or globals().get('val_model') is None) and os.path.exists(BEST_MODEL_PATH):
    val_model = YOLO(BEST_MODEL_PATH)

val_imgs = glob.glob(f'{LOCAL_DATASET}/val/*/*.jpg')

if not val_imgs:
    print("⚠️ Không có file test xác thực nào rơi vào Tập Val của Dataset.")
elif 'val_model' not in globals() or globals().get('val_model') is None:
    print("❌ Không quản lý được RAM: File học chưa lưu vào Global Cache.")
else:
    res = val_model.predict(source=val_imgs[0], save=True)
    res_path = os.path.join(res[0].save_dir, os.path.basename(val_imgs[0]))

    if os.path.exists(res_path):
        display(Image(filename=res_path, width=400))
    else:
        print("⚠️ Kết quả output bị hỏng hóc OS Colab.")

    names_map = res[0].names
    best_idx = res[0].probs.top1
    confidence = res[0].probs.top1conf.item()
    
    print(f"\n📝 Tổng kết đánh giá là: {names_map[best_idx].upper()} (Tin Cậy: {confidence:.2%})")

In [ ]:
# ============================================================
# Cell 6: Đóng gói kết quả ra Binary Tối Thượng ONNX
# ============================================================
import os
import shutil
from ultralytics import YOLO

BEST_MODEL_PATH = os.path.join(PROJECT_PATH, RUN_NAME, 'weights', 'best.pt')
if ('val_model' not in globals() or globals().get('val_model') is None) and os.path.exists(BEST_MODEL_PATH):
    val_model = YOLO(BEST_MODEL_PATH)

if 'val_model' not in globals() or globals().get('val_model') is None:
    print("❌ Lỗi môi trường Colab ko tìm được file .pt chính lâu.")
else:
    print("📦 Đẩy mã lệnh ONNX để IoT xải gọn nhẹ...")
    # Export tối ưu hoá Edge Computing
    onnx_path = val_model.export(format='onnx', imgsz=224, simplify=True)
    
    drive_onnx_path = os.path.join(PROJECT_PATH, RUN_NAME, 'weights', 'best.onnx')
    os.makedirs(os.path.dirname(drive_onnx_path), exist_ok=True)
    
    shutil.copy2(onnx_path, drive_onnx_path)
    print(f"✅ Đã khóa đóng gói thành công ở ổ DRIVE chính phủ (C:): {drive_onnx_path}")